# Module 5 — Computation Engine & Reliability (exercises)

Same real data as Module 4 (`learning/days/day5/eda/data/player_stats_2025-26_regular.csv`) via
DuckDB. Companion reading: `workbook.md`. Core cells: `duckdb`, `numpy`, stdlib — no new installs.


In [ ]:
import duckdb, time
import numpy as np
from pathlib import Path

here = Path.cwd()
root = None
for p in [here, *here.parents]:
    if (p / 'learning/days/day5/eda/data/player_stats_2025-26_regular.csv').exists():
        root = p; break
assert root is not None, 'Run this notebook from inside the statlas repo (any subdir).'

CSV = root / 'learning/days/day5/eda/data/player_stats_2025-26_regular.csv'
con = duckdb.connect()
con.execute(f"CREATE TABLE players AS SELECT * FROM read_csv_auto('{CSV.as_posix()}')")
n_rows = con.execute("SELECT COUNT(*) FROM players").fetchone()[0]
print(f"{n_rows} player-rows loaded")


## Exercise 1 — Vectorized vs. row-at-a-time, felt not just read
Implement a naive Python loop-sum and a numpy vectorized sum over the same synthetic column
(scaled up well past the real 500-ish rows so the crossover is visible). Time both.

In [ ]:
def row_at_a_time_sum(values):
    total = 0.0
    for v in values:          # per-element Python-level dispatch, like a Volcano iterator's next()
        total += v
    return total

def vectorized_sum(values):
    return float(np.sum(values))   # one C-level pass, no per-element Python dispatch

sizes = [10_000, 100_000, 1_000_000]
results = []
for n in sizes:
    data = np.random.default_rng(0).random(n)
    t0 = time.perf_counter(); s1 = row_at_a_time_sum(data); t1 = time.perf_counter()
    t2 = time.perf_counter(); s2 = vectorized_sum(data);      t3 = time.perf_counter()
    assert abs(s1 - s2) < 1e-6
    row_ms, vec_ms = (t1 - t0) * 1000, (t3 - t2) * 1000
    results.append((n, row_ms, vec_ms))
    print(f"n={n:>9,}  row-at-a-time={row_ms:9.2f} ms   vectorized={vec_ms:7.3f} ms   "
          f"speedup={row_ms/max(vec_ms,1e-9):8.1f}x")

# TODO: does the speedup ratio grow, shrink, or stay flat as n grows? Why?
assert results[-1][1] > results[-1][2], "Vectorized should be faster at the largest size"


## Exercise 2 — Window-function drills
(a) rank scorers, (b) team-relative average via `PARTITION BY`, (c) a leaderboard with a
**min-games qualifier** — reproduce the repo's actual reliability rule on real data.

In [ ]:
# (a) RANK() OVER
top_scorers = con.execute("""
    SELECT PLAYER_NAME, PTS, RANK() OVER (ORDER BY PTS DESC) AS pts_rank
    FROM players ORDER BY pts_rank LIMIT 5
""").df()
print(top_scorers)

# (b) AVG() OVER (PARTITION BY ...)
team_relative = con.execute("""
    SELECT PLAYER_NAME, TEAM_ABBREVIATION, PTS,
           AVG(PTS) OVER (PARTITION BY TEAM_ABBREVIATION) AS team_avg_pts,
           PTS - AVG(PTS) OVER (PARTITION BY TEAM_ABBREVIATION) AS above_team_avg
    FROM players ORDER BY above_team_avg DESC LIMIT 5
""").df()
print(team_relative)

# (c) TODO: leaderboard with a min-games qualifier (try QUALIFY; DuckDB supports it)
qualified_leaders = con.execute("""
    SELECT PLAYER_NAME, GP, STL, RANK() OVER (ORDER BY STL DESC) AS stl_rank
    FROM players
    WHERE GP >= 58
    QUALIFY stl_rank <= 5
""").df()
print(qualified_leaders)

unqualified_leaders = con.execute("""
    SELECT PLAYER_NAME, GP, STL FROM players ORDER BY STL DESC LIMIT 5
""").df()

# Assert the qualifier actually changes the leaderboard (a low-GP player gets excluded)
assert set(qualified_leaders['PLAYER_NAME']) != set(unqualified_leaders['PLAYER_NAME']),        "Expected the min-games qualifier to change who's on the board"
print("Unqualified top-5 GP:", list(unqualified_leaders['GP']))
print("Qualified   top-5 GP:", list(qualified_leaders['GP']))


## Exercise 3 — `assert_safe`, with a `LIMIT` ceiling
A second, independent pass at the validator (see M4 Exercise 3) that additionally caps result
size — a computation-engine-specific concern (a runaway unbounded query shouldn't take the process down).

In [ ]:
import re

MAX_LIMIT = 1000
FORBIDDEN_KEYWORDS = {"INSERT", "UPDATE", "DELETE", "DROP", "ALTER", "CREATE", "ATTACH",
                       "COPY", "PRAGMA", "EXEC", "EXECUTE", "GRANT", "TRUNCATE"}

def assert_safe_bounded(sql: str) -> bool:
    stripped = sql.strip().rstrip(';')
    if ';' in stripped:
        return False
    if not re.match(r'^\s*SELECT\b', stripped, re.IGNORECASE):
        return False
    tokens = set(re.findall(r'[A-Za-z_]+', stripped.upper()))
    if tokens & FORBIDDEN_KEYWORDS:
        return False
    # TODO: enforce a LIMIT ceiling -- if there's no LIMIT at all, treat as unbounded (reject);
    # if there is one, it must be <= MAX_LIMIT
    m = re.search(r'\bLIMIT\s+(\d+)', stripped, re.IGNORECASE)
    if m is None:
        return False
    if int(m.group(1)) > MAX_LIMIT:
        return False
    return True

cases = [
    ("SELECT * FROM players LIMIT 10", True),
    ("SELECT * FROM players LIMIT 5000", False),   # over the ceiling
    ("SELECT * FROM players", False),               # unbounded -- no LIMIT at all
    ("SELECT * FROM players; DROP TABLE players", False),
]
for sql, expected in cases:
    got = assert_safe_bounded(sql)
    assert got == expected, f"{sql!r}: expected {expected}, got {got}"
print("assert_safe_bounded: all cases passed.")


## Exercise 4 — Execution-guided decoding, simulated
Run a generate → try-execute → catch → retry loop over a list of candidate SQL strings, some of
which have intentional bugs. Measure before/after success rate.

In [ ]:
candidates = [
    "SELECT PLAYER_NAME, PTS FROM players ORDER BY PTS DESC LIMIT 5",          # good
    "SELECT PLAYER_NAME, PTSS FROM players ORDER BY PTS DESC LIMIT 5",         # bad column name
    "SELECT PLAYER_NAME, STL FROM players ORDER BY STL DESC LIMIT 5",          # good
    "SELECT PLAYER_NAME FROM players WHERE GP => 58",                          # syntax error
    "SELECT PLAYER_NAME, AST FROM players ORDER BY AST DESC LIMIT 5",          # good
]

def try_execute(sql, conn):
    try:
        return conn.execute(sql).fetchall(), None
    except Exception as e:
        return None, str(e)

# naive: no retry, just try each candidate once
naive_successes = sum(1 for c in candidates if try_execute(c, con)[1] is None)

# TODO: execution-guided retry -- on failure, apply a cheap fix-up (here: nothing fancy, just
# demonstrate the catch-and-classify step a real system would feed back into regeneration)
fixed = 0
for c in candidates:
    result, err = try_execute(c, con)
    if err is not None:
        print(f"CAUGHT before it shipped: {err.splitlines()[0][:80]}  <- {c}")
    else:
        fixed += 1

print(f"\nnaive success: {naive_successes}/{len(candidates)}  "
      f"(execution-guided decoding's job: catch the other {len(candidates)-naive_successes} "
      f"BEFORE they reach the user, not after)")
assert naive_successes == 3, "3 of the 5 candidates should execute cleanly"


## Exercise 5 — Self-consistency voting
Given *k* candidate SQL queries answering the same question, execute all and take a majority vote
over the **numeric result**. Show one case where voting fixes a bad candidate.

In [ ]:
from collections import Counter

# 3 candidates answering "who has the most points, and how many?" -- 2 agree, 1 is subtly wrong
same_question_candidates = [
    "SELECT PLAYER_NAME, PTS FROM players ORDER BY PTS DESC LIMIT 1",
    "SELECT PLAYER_NAME, PTS FROM players ORDER BY PTS DESC LIMIT 1",   # duplicate reasoning path
    "SELECT PLAYER_NAME, PTS FROM players WHERE GP >= 70 ORDER BY PTS DESC LIMIT 1",  # different filter -> maybe different answer
]

answers = [tuple(con.execute(c).fetchone()) for c in same_question_candidates]
vote = Counter(answers).most_common(1)[0]
print("candidate answers:", answers)
print("majority vote:", vote[0], f"({vote[1]}/{len(answers)} agree)")

# TODO: construct a case where ALL candidates share the same blind spot (e.g. all three forget
# the min-games qualifier) so majority vote does NOT fix the error -- self-consistency only
# helps when errors are *independent*, not systematic


## Math problems
See `workbook.md` for full statements. Verify Problems 1 and 3 here.

In [ ]:
# Problem 1 -- big-O / constant-factor argument
n, c, w, B = 500_000, 50, 1, 2000
row_total = n * (w + c)
vec_total = (n / B) * c + n * w
print(f"row-at-a-time total: {row_total:,}")
print(f"vectorized total:    {vec_total:,.0f}")
print(f"speedup ratio:       {row_total / vec_total:.1f}x")
assert vec_total < row_total

# Problem 3 -- self-consistency probability, brute-force enumeration for k=5
from itertools import product
def majority_prob_bruteforce(k, p):
    need = k // 2 + 1
    total = 0.0
    for outcome in product([0, 1], repeat=k):   # 1 = correct, 0 = wrong
        prob = 1.0
        for bit in outcome:
            prob *= p if bit == 1 else (1 - p)
        if sum(outcome) >= need:
            total += prob
    return total

for p in (0.5, 0.6, 0.7, 0.8):
    print(f"p={p}: majority-of-5 correct with prob {majority_prob_bruteforce(5, p):.3f}")


## Definition of done
- [ ] Vectorized-vs-loop benchmark shows a measured, real speedup at n=1,000,000
- [ ] `QUALIFY` STL leaderboard demonstrably changes the top-5 vs. the unqualified version (a 3-game small-sample "leader" gets excluded)
- [ ] `assert_safe_bounded` passes all 4 test cases (LIMIT ceiling enforced)
- [ ] Execution-guided loop catches exactly the 2 bugged candidates
- [ ] Math Problems 1 and 3 verified in code above
